In [1]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
transformers.__version__

'4.56.2'

In [2]:
model_name = "Qwen/Qwen3-0.6B"

In [3]:
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [4]:
from pathlib import Path

In [13]:
prompt = Path('case1/0.prompt').read_text()
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)
print(text)


<|im_start|>user
### Tools

The following JSON describes the tools that you have, what each tool is for, and which arguments you need to use each tool:
{
  "tools": [
    {
      "name": "get_current_time",
      "description": "Get current time in a specific timezones",
      "inputSchema": {
        "type": "object",
        "properties": {
          "timezone": {
            "type": "string",
            "description": "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Asia/Seoul' as local timezone if no timezone provided by the user."
          }
        },
        "required": [
          "timezone"
        ]
      }
    },
    {
      "name": "convert_time",
      "description": "Convert time between timezones",
      "inputSchema": {
        "type": "object",
        "properties": {
          "source_timezone": {
            "type": "string",
            "description": "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Asia/Seoul' as l

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

In [11]:
# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
print(thinking_content)
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
print(content)

<think>
Okay, let's tackle the user's request. They want the current time in San Francisco and then convert that time to Seoul time.

First, I need to use the get_current_time tool. The input requires the timezone. The user specified San Francisco, which is 'America/San Francisco' as the source timezone. So I'll call that tool with that timezone.

Next, after getting the current time, I need to convert it from San Francisco to Seoul. The convert_time tool needs the source, target timezones, and the time in 24-hour format. The user mentioned converting the current time, so I should use the time from the first tool's response as the input for the second tool.

Wait, the user didn't specify the exact time yet. So the first tool's output will be the current time in San Francisco. Then, the second tool will take that time, convert it to Seoul. So I need to make sure that the input for the second tool includes the time from the first call.

So the steps are: call get_current_time with San Fr